In [28]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/omarboukellalfsfl/avis-ibis-maroc-fin/avis_ibis_maroc_final.csv
/kaggle/input/datasets/omarboukellalfsfl/df-absa-bertopic/df_absa_bertopic.csv
/kaggle/input/datasets/omarboukellalfsfl/validation-manuelle-200-2-annotee/validation_manuelle_200_2_annotee.xlsx
/kaggle/input/datasets/omarboukellalfsfl/validation-manuelle-200/validation_manuelle_200_annotee.xlsx


# **Importation des données**

In [6]:
import pandas as pd
import numpy as np

# Chemin complet du fichier
file_path = '/kaggle/input/datasets/omarboukellalfsfl/avis-ibis-maroc-fin/avis_ibis_maroc_final.csv'

# Importer les données
df = pd.read_csv(file_path)

# Afficher un aperçu
print(" Données importées avec succès !")
print(f"\n Dimension : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"\n Colonnes disponibles :")
print(list(df.columns))

# Aperçu des 5 premières lignes
print("\n Aperçu des données :")
print(df.head())

# Informations sur les types de données
print("\n Informations sur le DataFrame :")
print(df.info())

# Statistiques de base sur les colonnes numériques
print("\n Statistiques descriptives :")
print(df.describe())

 Données importées avec succès !

 Dimension : 30494 lignes, 11 colonnes

 Colonnes disponibles :
['source', 'hotel_name', 'hotel_city', 'auteur', 'note', 'titre', 'texte', 'date_publication', 'date_sejour', 'type_voyage', 'langue']

 Aperçu des données :
            source             hotel_name  hotel_city               auteur  \
0  tripadvisor.com  Ibis Abdelmoumen Casa  Casablanca       Luis Antonio B   
1  tripadvisor.com  Ibis Abdelmoumen Casa  Casablanca                 Atif   
2  tripadvisor.com  Ibis Abdelmoumen Casa  Casablanca  Paradise00617377813   
3  tripadvisor.com  Ibis Abdelmoumen Casa  Casablanca            pascal136   
4  tripadvisor.com  Ibis Abdelmoumen Casa  Casablanca             khalil Z   

   note                                              titre  \
0   1.0            202511 - Terrible lodging in Casablanca   
1   3.0                            Budget hotel good value   
2   5.0  They spare no effort for your comfort without ...   
3   5.0                    

In [2]:
# Ou avec le nombre d'avis par hôtel
print("\n📊 Nombre d'avis par hôtel :")
print(df['hotel_name'].value_counts())


📊 Nombre d'avis par hôtel :
hotel_name
Ibis Casablanca City Center    3991
Ibis Marrakech Centre Gare     3552
Ibis Tanger City Center        3061
Ibis Fès                       2875
Ibis Casa Voyageurs            2288
Ibis Ouarzazate Centre         1997
Ibis Rabat Agdal               1950
Ibis Marrakech Palmeraie       1819
Ibis El Jadida                 1482
Ibis Casablanca Nearshore      1389
Ibis Abdelmoumen Casa          1295
Ibis Meknès                    1277
Ibis Casa Sidi Maarouf         1229
Ibis Mohammedia                 882
Ibis Oujda                      753
Ibis Marrakech centre gare      511
Ibis El jadida                  143
Name: count, dtype: int64


In [7]:
# Fusionner les doublons d'hôtels (casse différente)
df['hotel_name'] = df['hotel_name'].replace({
    'Ibis Marrakech centre gare': 'Ibis Marrakech Centre Gare',
    'Ibis El jadida': 'Ibis El Jadida'
})

# Vérifier le résultat
print("📊 Nombre d'avis par hôtel après fusion :")
print(df['hotel_name'].value_counts())
print(df['type_voyage'].value_counts())


📊 Nombre d'avis par hôtel après fusion :
hotel_name
Ibis Marrakech Centre Gare     4063
Ibis Casablanca City Center    3991
Ibis Tanger City Center        3061
Ibis Fès                       2875
Ibis Casa Voyageurs            2288
Ibis Ouarzazate Centre         1997
Ibis Rabat Agdal               1950
Ibis Marrakech Palmeraie       1819
Ibis El Jadida                 1625
Ibis Casablanca Nearshore      1389
Ibis Abdelmoumen Casa          1295
Ibis Meknès                    1277
Ibis Casa Sidi Maarouf         1229
Ibis Mohammedia                 882
Ibis Oujda                      753
Name: count, dtype: int64
type_voyage
Couple      7766
Famille     6916
Solo        6503
Affaires    4038
Groupe      3148
Name: count, dtype: int64


# **Nettoyage minimal du texte (compatible toutes langues)**

In [8]:
# ── 2. Installer regex ──
!pip install regex -q

# ── 3. Fonction nettoyage minimal — compatible toutes langues ──
import regex

def nettoyage_minimal(texte):
    if pd.isna(texte):
        return ""
    texte = str(texte)
    # Supprimer URLs
    texte = regex.sub(r'http\S+|www\S+', '', texte)
    # Supprimer emojis — garder toutes lettres unicode + ponctuation utile
    texte = regex.sub(r'[^\p{L}\p{N}\s\.,;:!?\'\"\(\)\-]', '', texte)
    # Supprimer espaces multiples
    texte = regex.sub(r' +', ' ', texte).strip()
    return texte

# ── 4. Appliquer sur texte brut ──
df['texte_clean'] = df['texte'].apply(nettoyage_minimal)

# ── 5. Vérification ──
print(f"\nAvis avec texte vide après nettoyage : {(df['texte_clean'] == '').sum()}")
print(f"\nExemple avant / après :")
idx = df[df['texte_clean'] != ''].index[0]
print(f"  AVANT : {df.loc[idx, 'texte'][:150]}")
print(f"  APRÈS : {df.loc[idx, 'texte_clean'][:150]}")

# ── 6. Vérification sur langues non-latines ──
for langue in ['Arabe', 'Russe', 'Coréen', 'Japonais']:
    subset = df[df['langue'] == langue]
    if len(subset) > 0:
        idx = subset.index[0]
        print(f"\n[{langue}]")
        print(f"  AVANT : {df.loc[idx, 'texte'][:100]}")
        print(f"  APRÈS : {df.loc[idx, 'texte_clean'][:100]}")


Avis avec texte vide après nettoyage : 12

Exemple avant / après :
  AVANT : The standard room of the Ibis is larger than expected, but has several shortcomings that compromise the stay. The bathroom is just reasonable: there i
  APRÈS : The standard room of the Ibis is larger than expected, but has several shortcomings that compromise the stay. The bathroom is just reasonable: there i

[Arabe]
  AVANT : الحقيقة أن موظفي الفندق لطفاء جدا وخصوصا في الاستقبال لكن سعر الغرفة وخدماتها لايتناسب مع سعرها ومع 
  APRÈS : الحقيقة أن موظفي الفندق لطفاء جدا وخصوصا في الاستقبال لكن سعر الغرفة وخدماتها لايتناسب مع سعرها ومع 

[Russe]
  AVANT : Отель прежде всего понравился удобством рассположения, вокруг много кафешек и ресторанов с традицион
  APRÈS : Отель прежде всего понравился удобством рассположения, вокруг много кафешек и ресторанов с традицион

[Coréen]
  AVANT : 가격대비 좋았어요
  APRÈS : 가격대비 좋았어요

[Japonais]
  AVANT : 早朝のTangier 行きの汽車に乗るため、立地だけで選んだホテル。駅を利用するには便利だけど、そうじゃない場合は周りに外国人向けのレストランもないので

In [9]:
import pandas as pd

df_principal = df.copy()

# **Création de la variable saison**

In [11]:
import pandas as pd

df_principal = df.copy()

# Créer la colonne saison à partir de date_sejour
df_principal['date_sejour'] = pd.to_datetime(df_principal['date_sejour'], errors='coerce')

def get_saison(date):
    if pd.isna(date):
        return 'Inconnu'
    m = date.month
    if m in [12, 1, 2]:
        return 'Hiver'
    elif m in [3, 4, 5]:
        return 'Printemps'
    elif m in [6, 7, 8]:
        return 'Été'
    else:
        return 'Automne'

df_principal['saison'] = df_principal['date_sejour'].apply(get_saison)

print(df_principal['saison'].value_counts())
print(df_principal.shape)

saison
Automne      7996
Été          7185
Printemps    7059
Hiver        6848
Inconnu      1406
Name: count, dtype: int64
(30494, 13)


In [12]:
print(df_principal['langue'].value_counts())

langue
Français       15713
Anglais         8655
Espagnol        1483
Italien          879
Allemand         836
Arabe            696
Néerlandais      693
Portugais        515
Japonais         227
Inconnu          187
Russe            147
Chinois          108
Turc              67
Coréen            47
Roumain           37
SO                25
Indonésien        22
NO                21
Catalan           18
Suédois           16
AF                14
LT                14
Polonais          10
Hébreu             9
TL                 8
HR                 5
Grec               5
ET                 5
Vietnamien         4
SK                 4
Thaï               3
CY                 3
Hongrois           3
LV                 3
Tchèque            3
Danois             2
SL                 2
Finnois            2
SW                 1
UR                 1
SQ                 1
Name: count, dtype: int64


# **Segmentation des avis en phrases (NLTK)**

Chaque avis est découpé en phrases individuelles avec NLTK, car un même avis peut contenir plusieurs sentiments sur des aspects différents. Chaque phrase conserve les métadonnées de l'avis d'origine (hôtel, langue, profil voyageur, saison).


In [13]:
import re
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

# ============================================================
# CONNECTEURS ADVERSATIFS — Multilingues
# ============================================================
CONNECTORS = [
    # Français
    r'\bmais\b', r'\bcependant\b', r'\btoutefois\b', r'\bninanmoins\b',
    r'\bnéanmoins\b', r'\bpar contre\b', r'\ben revanche\b',
    r'\bmalgré\b', r'\bsauf que\b', r'\bpourtant\b', r'\bor\b',
    # Anglais
    r'\bbut\b', r'\bhowever\b', r'\balthough\b', r'\bthough\b',
    r'\byet\b', r'\bnevertheless\b', r'\bnonetheless\b',
    r'\bdespite\b', r'\beven though\b', r'\bwhile\b', r'\bwhereas\b',
    r'\bunfortunately\b', r'\bexcept\b',
    # Espagnol
    r'\bpero\b', r'\bsin embargo\b', r'\baunque\b', r'\bno obstante\b',
    r'\ba pesar de\b', r'\bsalvo\b',
    # Arabe
    r'\bلكن\b', r'\bولكن\b', r'\bغير أن\b', r'\bإلا أن\b',
    r'\bرغم\b', r'\bبيد أن\b', r'\bمع ذلك\b', r'\bإلا\b',
    # Italien
    r'\bma\b', r'\bperò\b', r'\btuttavia\b', r'\bsebbene\b',
    r'\bnonostante\b', r'\beppure\b',
    # Allemand
    r'\baber\b', r'\bjedoch\b', r'\bobwohl\b', r'\bdennoch\b',
    r'\btrotzdem\b', r'\ballerdings\b',
    # Portugais
    r'\bmas\b', r'\bporém\b', r'\bcontudo\b', r'\bentretanto\b',
    r'\bembora\b', r'\bapesar\b',
    # Néerlandais
    r'\bmaar\b', r'\bechter\b', r'\bhoewel\b', r'\btoch\b',
]

CONNECTOR_PATTERN = re.compile(
    r'(?<!\w)(' + '|'.join(CONNECTORS) + r')(?!\w)',
    flags=re.IGNORECASE
)

# ============================================================
# MOTS VIDES DE LIAISON — à exclure après split virgule
# (segments qui ne parlent d'aucun aspect)
# ============================================================
MOTS_VIDES_LIAISON = re.compile(
    r'^(et|or|ni|donc|ainsi|aussi|alors|avec|pour|par|sur|'
    r'and|or|so|also|with|for|'
    r'y|o|también|así|con|para|'
    r'und|oder|auch|so|mit|für|'
    r'e|o|anche|così|con|per|'
    r'en|of|ook|dus|met|voor)(\s+\w+)*$',
    flags=re.IGNORECASE
)

# ============================================================
# SPLIT SUR CONNECTEURS ADVERSATIFS
# ============================================================
def split_on_connectors(phrase, min_words=2):
    parts = CONNECTOR_PATTERN.split(phrase)
    segments = []
    i = 0
    while i < len(parts):
        if i + 1 < len(parts) and CONNECTOR_PATTERN.match(parts[i + 1] if i + 1 < len(parts) else ''):
            seg_avant = parts[i].strip()
            seg_apres = parts[i + 2].strip() if i + 2 < len(parts) else ''
            if seg_avant and len(seg_avant.split()) >= min_words:
                segments.append(seg_avant)
            if seg_apres and len(seg_apres.split()) >= min_words:
                segments.append(seg_apres)
            i += 3
        else:
            seg = parts[i].strip()
            if seg and len(seg.split()) >= min_words:
                segments.append(seg)
            i += 1
    return segments if segments else [phrase]

# ============================================================
# SPLIT SUR VIRGULE — NOUVEAU
# ============================================================
def split_on_comma(phrase, min_words=2):
    """
    Coupe sur virgule et point-virgule.
    Garde les segments >= min_words mots.
    Exclut les fragments de liaison sans contenu.
    """
    # Split sur virgule ET point-virgule
    segments = re.split(r'[,;]', phrase)
    result = []
    for seg in segments:
        seg = seg.strip()
        # Ignorer segments trop courts
        if len(seg.split()) < min_words:
            continue
        # Ignorer segments qui commencent par mots de liaison purs
        if MOTS_VIDES_LIAISON.match(seg) and len(seg.split()) <= 3:
            continue
        result.append(seg)
    return result if len(result) > 1 else [phrase]

# ============================================================
# LANGUE → NLTK
# ============================================================
langue_nltk = {
    'Français'    : 'french',
    'Anglais'     : 'english',
    'Espagnol'    : 'spanish',
    'Italien'     : 'italian',
    'Allemand'    : 'german',
    'Néerlandais' : 'dutch',
    'Portugais'   : 'portuguese',
    'Roumain'     : 'romanian',
    'Suédois'     : 'swedish',
    'Polonais'    : 'polish',
    'Danois'      : 'danish',
    'Finnois'     : 'finnish',
    'Tchèque'     : 'czech',
    'Hongrois'    : 'english',
    'Grec'        : 'greek',
    'Norvégien'   : 'norwegian',
    'Arabe'       : 'english',
    'Japonais'    : 'english',
    'Russe'       : 'english',
    'Chinois'     : 'english',
    'Turc'        : 'english',
    'Coréen'      : 'english',
    'Indonésien'  : 'english',
    'Catalan'     : 'spanish',
    'Hébreu'      : 'english',
    'Vietnamien'  : 'english',
    'Thaï'        : 'english',
}

# ============================================================
# SEGMENTATION COMPLÈTE — NLTK + Connecteurs + Virgule
# ============================================================
MIN_MOTS = 2  # ← réduit de 4 à 2

phrases_list = []
n_nltk_only  = 0
n_connector  = 0
n_comma      = 0

for _, row in df_principal.iterrows():
    lang_nltk = langue_nltk.get(row['langue'], 'english')
    texte = str(row['texte_clean'])

    # ── Étape 1 : split NLTK (sur . ! ?) ────────────────────
    try:
        phrases_nltk = sent_tokenize(texte, language=lang_nltk)
    except Exception:
        phrases_nltk = texte.split('.')
        phrases_nltk = [p.strip() for p in phrases_nltk if len(p.strip()) > 5]

    phrases_nltk = [p for p in phrases_nltk if len(p.split()) >= MIN_MOTS]

    for phrase_nltk in phrases_nltk:

        # ── Étape 2 : split sur connecteurs adversatifs ──────
        segments_conn = split_on_connectors(phrase_nltk, min_words=MIN_MOTS)
        if len(segments_conn) > 1:
            n_connector += len(segments_conn) - 1

        # ── Étape 3 : split sur virgule pour chaque segment ──
        final_segments = []
        for seg in segments_conn:
            sub = split_on_comma(seg, min_words=MIN_MOTS)
            if len(sub) > 1:
                n_comma += len(sub) - 1
            final_segments.extend(sub)

        if len(final_segments) == 1:
            n_nltk_only += 1

        for seg in final_segments:
            phrases_list.append({
                'hotel_name'  : row['hotel_name'],
                'hotel_city'  : row['hotel_city'],
                'date_sejour' : row['date_sejour'],
                'saison'      : row['saison'],
                'langue'      : row['langue'],
                'type_voyage' : row['type_voyage'],
                'source'      : row['source'],
                'note'        : row['note'],
                'phrase'      : seg,
            })

df_phrases = pd.DataFrame(phrases_list)

# ============================================================
# STATS
# ============================================================
print(f"Reviews              : {len(df_principal):,}")
print(f"Phrases NLTK seules  : {n_nltk_only:,}")
print(f"Coupures connecteurs : +{n_connector:,}")
print(f"Coupures virgules    : +{n_comma:,}")       # ← nouveau
print(f"Phrases totales      : {len(df_phrases):,}")
print(f"Moyenne / review     : {len(df_phrases)/len(df_principal):.1f}")
print(f"\nDistribution par langue :")
print(df_phrases['langue'].value_counts().head(10))

# ── TEST sur la phrase problématique ────────────────────────
print(f"\nTEST virgule multi-aspects :")
test1 = "Hôtel très bien situé, petit déjeuner trop bon, chambre confortable, personnel aimable, bon rapport qualité prix"
print(f"  Avant : {test1}")
for s in split_on_comma(test1):
    print(f"  → {s}")

print(f"\nTEST connecteur arabe :")
test2 = "الحقيقة أن موظفي الفندق لطفاء جدا وخصوصا في الاستقبال لكن سعر الغرفة وخدماتها لايتناسب مع سعرها"
print(f"  Avant : {test2}")
for s in split_on_connectors(test2):
    print(f"  → {s}")

print(f"\nTEST phrase courte 2 mots :")
test3 = "chambre propre"
segs = split_on_comma(test3, min_words=2)
print(f"  '{test3}' → {segs}")

Reviews              : 30,494
Phrases NLTK seules  : 57,662
Coupures connecteurs : +10,891
Coupures virgules    : +46,016
Phrases totales      : 147,890
Moyenne / review     : 4.8

Distribution par langue :
langue
Français       71379
Anglais        51490
Espagnol        7390
Italien         4709
Allemand        4308
Portugais       3060
Néerlandais     2504
Russe           1322
Arabe            838
Turc             228
Name: count, dtype: int64

TEST virgule multi-aspects :
  Avant : Hôtel très bien situé, petit déjeuner trop bon, chambre confortable, personnel aimable, bon rapport qualité prix
  → Hôtel très bien situé
  → petit déjeuner trop bon
  → chambre confortable
  → personnel aimable
  → bon rapport qualité prix

TEST connecteur arabe :
  Avant : الحقيقة أن موظفي الفندق لطفاء جدا وخصوصا في الاستقبال لكن سعر الغرفة وخدماتها لايتناسب مع سعرها
  → الحقيقة أن موظفي الفندق لطفاء جدا وخصوصا في الاستقبال
  → سعر الغرفة وخدماتها لايتناسب مع سعرها

TEST phrase courte 2 mots :
  'chamb

# **Encodage des phrases en vecteurs (SentenceTransformer)**
Chaque phrase est transformée en vecteur numérique de 768 dimensions grâce au modèle multilingue paraphrase-multilingual-mpnet-base-v2, capable de représenter le sens sémantique d'une phrase dans toutes les langues du corpus.


In [14]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device : {device}")

model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2', device=device)

# fp16 directement sur le modèle
if device == 'cuda':
    model = model.half()

phrases = df_phrases['phrase'].tolist()
print(f"Encodage de {len(phrases)} phrases...")

embeddings = model.encode(
    phrases,
    batch_size=512,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f"Shape embeddings : {embeddings.shape}")

np.save('embeddings_phrases.npy', embeddings)
print("Embeddings sauvegardés ✅")

Device : cuda


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encodage de 147890 phrases...


Batches:   0%|          | 0/289 [00:00<?, ?it/s]

Shape embeddings : (147890, 768)
Embeddings sauvegardés ✅


In [15]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 3.9 MB/s eta 0:00:0000:01


# **Détection des aspects avec BERTopic (Zero-Shot)**
Chaque phrase est assignée à l'un des 11 aspects prédéfinis grâce à BERTopic en mode zero-shot — sans entraînement supervisé. Les mots-clés de chaque aspect sont définis en plusieurs langues (français, anglais, espagnol, arabe, russe) pour couvrir l'ensemble du corpus multilingue.


In [21]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import numpy as np

# ============================================================
# CHARGER EMBEDDINGS (déjà calculés)
# ============================================================

embeddings = np.load('embeddings_phrases.npy')
phrases    = df_phrases['phrase'].tolist()

print(f"✓ Embeddings chargés : {embeddings.shape}")
assert len(embeddings) == len(phrases), \
    f"⚠ Mismatch : {len(embeddings)} embeddings vs {len(phrases)} phrases"

embedding_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# ============================================================
# UMAP / HDBSCAN / VECTORIZER  (inchangés)
# ============================================================

umap_model = UMAP(
    n_neighbors  = 15,
    n_components = 5,
    min_dist     = 0.0,
    metric       = 'cosine',
    random_state = 42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size          = 15,
    min_samples               = 5,
    cluster_selection_epsilon = 0.05,
    metric                    = 'euclidean',
    prediction_data           = True,
)

STOP_WORDS_CUSTOM = [
    'the','to','a','an','and','or','is','was','were','are','be','been',
    'it','its','this','that','i','we','my','our','he','she','they',
    'in','on','at','for','of','with','by','from','as','have','had',
    'not','no','but','so','if','then','than','very','also','just',
    'le','la','les','un','une','des','du','de','et','est','en','au','aux',
    'je','il','elle','nous','on','ce','se','sa','son','ses','qui','que',
    'ne','pas','plus','très','aussi','mais','si','car','donc','ou','où',
    'y','par','sur','sous','avec','pour','dans','chez',
    'el','los','las','del','al','y','es','se','su','sus','con','por','para','que','no',
    'der','die','das','ein','eine','und','ist','zu','mit','von','auf','an','im','für','nicht','auch','sie','er',
    'il','lo','gli','di','da','su','per','con','non','è','e',
    'de','het','een','van','en','in','is','dat','op','met','aan','voor','zijn',
    'ibis','hotel','hôtel','morocco','maroc','marruecos',
]

vectorizer_model = CountVectorizer(
    ngram_range = (1, 2),
    stop_words  = STOP_WORDS_CUSTOM,
    min_df      = 5,
    max_df      = 0.90,
)

# ============================================================
# ZEROSHOT  (inchangé)
# ============================================================

zeroshot_topic_list = [
    "personnel accueil réception employé équipe sympathique aimable serviable "
    "staff reception employee friendly helpful team "
    "personal recepción amable servicial "
    "Personal Rezeption freundlich hilfsbereit "
    "personale reception gentile disponibile "
    "personeel receptie vriendelijk behulpzaam "
    "موظف استقبال ودود مساعد خدمة",

    "service prestation qualité check-in check-out prise en charge "
    "service quality room service check-in check-out "
    "servicio calidad atención "
    "Service Dienstleistung Qualität "
    "خدمة جودة الخدمة رعاية",

    "chambre lit confort propreté hygiène propre sale literie salle de bain "
    "room bed cleanliness clean dirty bathroom comfort bedding "
    "habitación cama limpieza limpio sucio comodidad "
    "Zimmer Bett Sauberkeit sauber schmutzig Badezimmer "
    "غرفة سرير نظافة نظيف قذر حمام",

    "restauration petit-déjeuner buffet nourriture repas restaurant dîner déjeuner "
    "food breakfast buffet restaurant meal dinner lunch dining "
    "desayuno comida restaurante buffet cena "
    "Frühstück Essen Restaurant Mahlzeit "
    "إفطار طعام مطعم وجبة بوفيه",

    "localisation emplacement situation quartier centre-ville transport proximité accès "
    "location area neighborhood transport city center proximity access "
    "ubicación centro transporte acceso "
    "Lage Standort Innenstadt Transport Nähe "
    "موقع وسط المدينة مواصلات",

    "expérience séjour impression globalement recommander satisfaction agréable "
    "experience stay overall recommend satisfaction pleasant "
    "experiencia estancia recomendable satisfacción "
    "Erfahrung Aufenthalt empfehlen Zufriedenheit "
    "تجربة إقامة توصية رضا ممتع",

    "prix rapport qualité-prix tarif cher abordable valeur coût économique "
    "price value for money cost expensive affordable cheap worth "
    "precio relación calidad-precio caro económico "
    "Preis Preis-Leistungs-Verhältnis teuer günstig "
    "سعر قيمة غالي رخيص ميسور",

    "équipements piscine parking salle de sport spa activités installations "
    "amenities pool parking gym spa facilities activities "
    "instalaciones piscina parking gimnasio spa "
    "Einrichtungen Pool Parkplatz Fitnessstudio Spa "
    "مرافق مسبح موقف سيارات نادي رياضي",

    "bruit nuisance sonore calme silencieux bruyant sommeil nuit "
    "noise noisy quiet sleep loud disturbance sound night "
    "ruido ruidoso tranquilo silencioso sueño "
    "Lärm laut ruhig leise Schlaf "
    "ضوضاء صاخب هادئ نوم",

    "température climatisation chauffage chaud froid confort thermique "
    "temperature air conditioning heating hot cold AC "
    "temperatura aire acondicionado calor frío "
    "Temperatur Klimaanlage Heizung heiß kalt "
    "حرارة تكييف تدفئة حار بارد",

    "wifi internet connexion réseau débit vitesse lent rapide "
    "wifi internet connection network speed slow fast "
    "wifi internet conexión red velocidad "
    "WLAN Internet Verbindung Geschwindigkeit "
    "واي فاي انترنت اتصال شبكة",
]

# ============================================================
# BERTOPIC
# ============================================================

topic_model = BERTopic(
    embedding_model         = embedding_model,
    umap_model              = umap_model,
    hdbscan_model           = hdbscan_model,
    vectorizer_model        = vectorizer_model,
    zeroshot_topic_list     = zeroshot_topic_list,
    zeroshot_min_similarity = 0.45,
    min_topic_size          = 15,
    verbose                 = True,
)

print(f"\nFit BERTopic sur {len(phrases):,} phrases...")
topics, probs = topic_model.fit_transform(phrases, embeddings)

# ============================================================
# RÉSULTATS
# ============================================================

topic_info = topic_model.get_topic_info()
n_outliers = topics.count(-1)
n_topics   = len(topic_info) - 1

print(f"\n{'='*50}")
print(f"  Topics trouvés      : {n_topics}")
print(f"  Phrases totales     : {len(topics):,}")
print(f"  Topic -1 (outliers) : {n_outliers:,}  ({n_outliers/len(topics)*100:.1f}%)")

if n_outliers / len(topics) > 0.30:
    print(f"  ⚠ > 30% outliers → baisser zeroshot_min_similarity à 0.40")
else:
    print(f"  ✓ Taux outliers acceptable")

print(f"\n{topic_info.head(16)[['Topic','Count','Name']].to_string(index=False)}")

✓ Embeddings chargés : (147890, 768)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-28 23:27:43,003 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm



Fit BERTopic sur 147,890 phrases...


2026-05-28 23:31:59,937 - BERTopic - Dimensionality - Completed ✓
2026-05-28 23:31:59,941 - BERTopic - Zeroshot Step 1 - Finding documents that could be assigned to either one of the zero-shot topics
2026-05-28 23:32:01,735 - BERTopic - Zeroshot Step 1 - Completed ✓
2026-05-28 23:32:47,126 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-28 23:32:56,099 - BERTopic - Cluster - Completed ✓
2026-05-28 23:32:56,100 - BERTopic - Zeroshot Step 2 - Combining topics from zero-shot topic modeling with topics from clustering...
2026-05-28 23:32:56,450 - BERTopic - Zeroshot Step 2 - Completed ✓
2026-05-28 23:32:56,453 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-28 23:32:59,799 - BERTopic - Representation - Completed ✓



  Topics trouvés      : 989
  Phrases totales     : 147,890
  Topic -1 (outliers) : 32,753  (22.1%)
  ✓ Taux outliers acceptable

 Topic  Count                                                                                                                                                                                                                                                                                                                 Name
    -1  32753                                                                                                                                                                                                                                                                                                      -1_me_us_ai_ils
     0  10661 personnel accueil réception employé équipe sympathique aimable serviable staff reception employee friendly helpful team personal recepción amable servicial Personal Rezeption freundlich hilfsbereit personale r

In [17]:
# Après fit_transform, exécute ceci
for i in range(11):
    try:
        words = [w for w, _ in topic_model.get_topic(i)[:5]]
        print(f"Topic {i} → {words}")
    except:
        print(f"Topic {i} → introuvable")

Topic 0 → ['personnel', 'staff', 'friendly', 'accueillant', 'serviable']
Topic 1 → ['service', 'qualité', 'bon rapport', 'rapport qualité', 'qualité prix']
Topic 2 → ['clean', 'bain', 'chambres', 'chambre', 'propre']
Topic 3 → ['buffet', 'déjeuner', 'petit déjeuner', 'restaurant', 'petit']
Topic 4 → ['centre ville', 'ville', 'centre', 'gare', 'tram']
Topic 5 → ['séjour', 'séjour agréable', 'bon séjour', 'experience', 'agréable séjour']
Topic 6 → ['qualité prix', 'rapport qualité', 'rapport', 'prix', 'bon rapport']
Topic 7 → ['piscine', 'pool', 'piscina', 'swimming pool', 'swimming']
Topic 8 → ['bruit', 'bruyant', 'noise', 'noisy', 'nuit']
Topic 9 → ['climatisation', 'air', 'air conditioning', 'conditioning', 'chauffage']
Topic 10 → ['wifi', 'fi', 'wi fi', 'wi', 'connexion']


# **Mapping des topics vers les 11 aspects**
Les 194 topics détectés par BERTopic sont manuellement mappés aux 11 aspects de l'étude. Chaque topic est analysé par ses mots-clés dominants puis rattaché à l'aspect correspondant. Les phrases sans topic valide (topic = -1) restent non assignées et seront traitées à l'étape suivante.


In [22]:
import pandas as pd
import numpy as np

# ============================================================
# MAPPING TOPIC → ASPECT  (topics 0-10 = zeroshot)
# ============================================================

TOPIC_TO_ASPECT = {
    0  : 'personnel',
    1  : 'service',
    2  : 'chambre_proprete',
    3  : 'restauration',
    4  : 'localisation',
    5  : 'experience',
    6  : 'prix',
    7  : 'equipements',
    8  : 'bruit',
    9  : 'temperature',
    10 : 'wifi',
    # Topics 11+ → cosine fallback
}

ASPECT_LABELS = {
    'personnel'        : 'Personnel',
    'service'          : 'Service',
    'chambre_proprete' : 'Chambre & Propreté',
    'restauration'     : 'Restauration',
    'localisation'     : 'Localisation',
    'experience'       : 'Expérience & Valeur',
    'prix'             : 'Rapport qualité-prix',
    'equipements'      : 'Équipements & Activités',
    'bruit'            : 'Bruit & Qualité du sommeil',
    'temperature'      : 'Confort thermique / Température',
    'wifi'             : 'Internet & Wi-Fi',
}

# ============================================================
# APPLIQUER LE MAPPING
# ============================================================

df_phrases['topic']        = topics
df_phrases['aspect']       = df_phrases['topic'].map(TOPIC_TO_ASPECT)
df_phrases['aspect_label'] = df_phrases['aspect'].map(ASPECT_LABELS)

# Mots-clés BERTopic par topic
topic_info     = topic_model.get_topic_info()
topic_keywords = {}
for _, row in topic_info.iterrows():
    tid = row['Topic']
    if tid == -1:
        continue
    try:
        words = topic_model.get_topic(tid)
        topic_keywords[tid] = ", ".join([w for w, _ in words[:5]])
    except Exception:
        topic_keywords[tid] = ""

df_phrases['topic_keywords'] = df_phrases['topic'].map(topic_keywords).fillna("")

# avis_id
if 'avis_id' not in df_phrases.columns:
    df_phrases['avis_id'] = df_phrases.index

# ============================================================
# STATS
# ============================================================

total      = len(df_phrases)
assigned   = df_phrases['aspect'].notna().sum()
outliers   = (df_phrases['topic'] == -1).sum()
non_mappes = df_phrases['aspect'].isna().sum() - outliers

print(f"{'='*55}")
print(f"  RÉSULTATS MAPPING")
print(f"{'='*55}")
print(f"  Phrases totales              : {total:>8,}")
print(f"  Assignées (topics 0-10)      : {assigned:>8,}  ({assigned/total*100:.1f}%)")
print(f"  Topic -1 (outliers BERTopic) : {outliers:>8,}  ({outliers/total*100:.1f}%)")
print(f"  Topics 11+ non mappés        : {non_mappes:>8,}  ({non_mappes/total*100:.1f}%)")
print(f"{'─'*55}")
print(f"  → {outliers + non_mappes:,} phrases iront en cosine fallback")
print(f"{'='*55}")

print(f"\n  DISTRIBUTION PAR ASPECT")
print(f"  {'Aspect':<30} {'Phrases':>8}   {'%':>5}")
print(f"  {'─'*48}")
for key, label in ASPECT_LABELS.items():
    count = (df_phrases['aspect'] == key).sum()
    if count == 0:
        continue
    print(f"  {label:<30} {count:>8,}   {count/total*100:>4.1f}%")

  RÉSULTATS MAPPING
  Phrases totales              :  147,890
  Assignées (topics 0-10)      :   54,594  (36.9%)
  Topic -1 (outliers BERTopic) :   32,753  (22.1%)
  Topics 11+ non mappés        :   60,543  (40.9%)
───────────────────────────────────────────────────────
  → 93,296 phrases iront en cosine fallback

  DISTRIBUTION PAR ASPECT
  Aspect                          Phrases       %
  ────────────────────────────────────────────────
  Personnel                        10,661    7.2%
  Service                           2,860    1.9%
  Chambre & Propreté               21,471   14.5%
  Restauration                      8,003    5.4%
  Localisation                      3,163    2.1%
  Expérience & Valeur                 901    0.6%
  Rapport qualité-prix                763    0.5%
  Équipements & Activités           3,621    2.4%
  Bruit & Qualité du sommeil          816    0.6%
  Confort thermique / Température    1,433    1.0%
  Internet & Wi-Fi                    902    0.6%


# **Réassignation par similarité cosinus (Fallback)**
Les phrases non assignées par BERTopic (topic = -1) sont réassignées en calculant leur similarité cosinus avec les descriptions des 10 aspects. Si la similarité dépasse le seuil de 0.40, la phrase est rattachée à l'aspect le plus proche sémantiquement.


In [24]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ============================================================
# DESCRIPTIONS DES 11 ASPECTS
# ============================================================

ASPECT_DESCRIPTIONS = {
    'personnel'        : (
        "personnel accueil réception employé équipe sympathique aimable serviable poli "
        "staff reception employee friendly helpful team rude impoli check-in check-out "
        "personal recepción amable servicial موظف استقبال ودود مساعد"
    ),
    'service'          : (
        "service prestation qualité de service prise en charge room service "
        "service quality attentive professional efficient hospitality "
        "servicio calidad atención خدمة جودة الخدمة"
    ),
    'chambre_proprete' : (
        "chambre lit confort propreté hygiène propre sale literie salle de bain douche "
        "room bed cleanliness clean dirty bathroom shower comfort bedding renovation vieux "
        "habitación cama limpieza limpio sucio غرفة سرير نظافة نظيف قذر حمام"
    ),
    'restauration'     : (
        "restauration petit-déjeuner buffet nourriture repas restaurant dîner déjeuner "
        "food breakfast buffet restaurant meal dinner lunch waiter menu boisson bar "
        "desayuno comida restaurante إفطار طعام مطعم وجبة بوفيه"
    ),
    'localisation'     : (
        "localisation emplacement situation quartier centre-ville transport proximité accès "
        "gare train aéroport taxi plage médina marche pied "
        "location area neighborhood transport city center airport station "
        "موقع وسط المدينة مواصلات محطة"
    ),
    'experience'       : (
        "expérience séjour impression globalement recommander satisfaction agréable "
        "ensemble overall recommend pleasant wonderful excellent horrible "
        "experiencia estancia recomendable satisfacción "
        "تجربة إقامة توصية رضا ممتع"
    ),
    'prix'             : (
        "prix rapport qualité-prix tarif cher abordable valeur coût économique budget "
        "price value money cost expensive affordable cheap worth dirhams euros "
        "precio relación calidad-precio caro "
        "سعر قيمة غالي رخيص ميسور"
    ),
    'equipements'      : (
        "équipements piscine parking salle de sport spa gym activités installations "
        "ascenseur jardin terrasse télévision tv "
        "amenities pool parking gym spa facilities elevator garden terrace "
        "instalaciones piscina مرافق مسبح موقف سيارات نادي"
    ),
    'bruit'            : (
        "bruit nuisance sonore calme silencieux bruyant sommeil nuit agitée "
        "voisin couloir rue klaxon fête chien aboiement "
        "noise noisy quiet sleep loud disturb night sound neighbor "
        "ruido ruidoso tranquilo ضوضاء صاخب هادئ نوم"
    ),
    'temperature'      : (
        "température climatisation chauffage chaud froid confort thermique ventilateur "
        "air conditioning heating hot cold AC clim trop chaud "
        "temperatura aire acondicionado calor frío "
        "حرارة تكييف تدفئة حار بارد"
    ),
    'wifi'             : (
        "wifi internet connexion réseau débit vitesse lent rapide signal "
        "wi-fi connection network speed slow fast bandwidth "
        "واي فاي انترنت اتصال شبكة سرعة"
    ),
}

# ============================================================
# INITIALISER COLONNES SOURCE
# ============================================================

df_phrases['cosine_score']  = np.nan
df_phrases['source_aspect'] = np.where(df_phrases['aspect'].notna(), 'bertopic', None)

# ============================================================
# PHRASES NON ASSIGNÉES
# ============================================================

unassigned_mask = df_phrases['aspect'].isna()
n_unassigned    = unassigned_mask.sum()
print(f"Phrases à réassigner : {n_unassigned:,}")

# ── Embeddings des phrases non assignées ────────────────────
unassigned_idx  = df_phrases[unassigned_mask].index
unassigned_embeds = embeddings[unassigned_idx]

# ── Encoder les descriptions d'aspects ──────────────────────
aspect_names  = list(ASPECT_DESCRIPTIONS.keys())
aspect_embeds = embedding_model.encode(
    list(ASPECT_DESCRIPTIONS.values()),
    convert_to_numpy  = True,
    show_progress_bar = False,
)

# ============================================================
# SIMILARITÉ COSINUS
# ============================================================

THRESHOLD = 0.35

print("Calcul similarité cosinus...")
sims            = cosine_similarity(unassigned_embeds, aspect_embeds)
best_scores     = sims.max(axis=1)
best_aspect_idx = sims.argmax(axis=1)

assigned_count = 0
for i, idx in enumerate(unassigned_idx):
    score = best_scores[i]
    if score >= THRESHOLD:
        best_aspect = aspect_names[best_aspect_idx[i]]
        df_phrases.at[idx, 'aspect']        = best_aspect
        df_phrases.at[idx, 'aspect_label']  = ASPECT_LABELS[best_aspect]
        df_phrases.at[idx, 'cosine_score']  = round(float(score), 4)
        df_phrases.at[idx, 'source_aspect'] = 'cosine'
        assigned_count += 1

# ============================================================
# RÉSULTATS
# ============================================================

total          = len(df_phrases)
total_assigned = df_phrases['aspect'].notna().sum()
still_na       = df_phrases['aspect'].isna().sum()

print(f"\n{'='*55}")
print(f"  RÉSULTATS COSINUS FALLBACK")
print(f"{'='*55}")
print(f"  Phrases traitées          : {n_unassigned:>8,}")
print(f"  Réassignées (≥ {THRESHOLD})    : {assigned_count:>8,}  ({assigned_count/n_unassigned*100:.1f}%)")
print(f"  Encore non assignées      : {still_na:>8,}  ({still_na/total*100:.1f}%)")
print(f"{'─'*55}")
print(f"  Taux assignation total    : {total_assigned/total*100:.1f}%")
print(f"{'='*55}")

print(f"\n  DISTRIBUTION FINALE PAR ASPECT")
print(f"  {'Aspect':<30} {'Total':>7}  {'BERTopic':>9}  {'Cosinus':>8}")
print(f"  {'─'*58}")
for key, label in ASPECT_LABELS.items():
    n_total  = (df_phrases['aspect'] == key).sum()
    n_bert   = ((df_phrases['aspect'] == key) & (df_phrases['source_aspect'] == 'bertopic')).sum()
    n_cosine = ((df_phrases['aspect'] == key) & (df_phrases['source_aspect'] == 'cosine')).sum()
    if n_total == 0:
        continue
    print(f"  {label:<30} {n_total:>7,}  {n_bert:>9,}  {n_cosine:>8,}")

# ============================================================
# SAUVEGARDE
# ============================================================

cols = [c for c in [
    'avis_id', 'phrase', 'topic', 'topic_keywords',
    'aspect', 'aspect_label', 'source_aspect', 'cosine_score',
    'hotel_name', 'hotel_city', 'source', 'langue', 'note',
    'date_sejour', 'saison', 'type_voyage',
] if c in df_phrases.columns]

df_phrases[cols].to_csv('/kaggle/working/df_phrases_aspects_v4.csv', index=False, encoding='utf-8-sig')
print(f"\n✓ Sauvegardé → /kaggle/working/df_phrases_aspects_v4.csv")
print(f"  {total_assigned:,} phrases avec aspect  |  {still_na:,} sans aspect")

Phrases à réassigner : 63,866
Calcul similarité cosinus...

  RÉSULTATS COSINUS FALLBACK
  Phrases traitées          :   63,866
  Réassignées (≥ 0.35)    :   18,441  (28.9%)
  Encore non assignées      :   45,425  (30.7%)
───────────────────────────────────────────────────────
  Taux assignation total    : 69.3%

  DISTRIBUTION FINALE PAR ASPECT
  Aspect                           Total   BERTopic   Cosinus
  ──────────────────────────────────────────────────────────
  Personnel                       15,185     13,033     2,152
  Service                          8,680      6,843     1,837
  Chambre & Propreté              31,232     27,852     3,380
  Restauration                    12,731     11,023     1,708
  Localisation                    10,929      8,762     2,167
  Expérience & Valeur              6,072      3,305     2,767
  Rapport qualité-prix             2,160      1,428       732
  Équipements & Activités          7,403      5,882     1,521
  Bruit & Qualité du sommeil     

# **Classification du sentiment par phrase (XLM-RoBERTa)**
Chaque phrase est classifiée en positif / négatif / neutre avec le modèle joeddav/xlm-roberta-large-xnli, un modèle multilingue entraîné sur l'inférence de langage naturel. La classification intègre le contexte de l'aspect (aspect : chambre_confort]) pour affiner le sentiment détecté. 

In [25]:
from transformers import pipeline
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset

# ============================================================
# DATASET CLASS — pour pipeline optimisé
# ============================================================
class PhraseDataset(Dataset):
    def __init__(self, phrases):
        self.phrases = phrases
    def __len__(self):
        return len(self.phrases)
    def __getitem__(self, i):
        return self.phrases[i]

# ============================================================
# CHARGEMENT
# ============================================================
df_assigned = df_phrases[df_phrases['aspect'].notna()].copy().reset_index(drop=True)
print(f"Phrases à analyser : {len(df_assigned):,}")

# ============================================================
# MODÈLE — optimisé T4
# ============================================================
classifier = pipeline(
    "text-classification",
    model       = "cardiffnlp/xlm-roberta-base-sentiment-multilingual",
    device      = 0,
    truncation  = True,
    max_length  = 128,   # ← réduit de 512 à 128 (avis courts)
    top_k       = None,
    torch_dtype = torch.float16,
)

LABEL_MAP = {
    'positive': 'positive', 'negative': 'negative', 'neutral': 'neutral',
    'pos': 'positive',      'neg': 'negative',      'neu': 'neutral',
}

# ============================================================
# INFÉRENCE — batch 512 + Dataset class
# ============================================================
BATCH_SIZE = 512   # ← augmenté de 128 → 512 pour T4

dataset    = PhraseDataset(df_assigned['phrase'].fillna("").tolist())
sentiments = []
scores     = []
n_total    = len(df_assigned)

print(f"Analyse sentiment ({n_total:,} phrases, batch={BATCH_SIZE})...")

for out in classifier(dataset, batch_size=BATCH_SIZE):
    meilleur = max(out, key=lambda x: x['score'])
    label    = LABEL_MAP.get(meilleur['label'].lower(), meilleur['label'].lower())
    sentiments.append(label)
    scores.append(round(meilleur['score'], 4))

    # Progress
    n_done = len(sentiments)
    if n_done % 10000 == 0:
        print(f"  {n_done:>7,} / {n_total:,}  ({n_done/n_total*100:.1f}%)")

df_assigned['sentiment']       = sentiments
df_assigned['score_sentiment'] = scores

# ============================================================
# RÉSULTATS
# ============================================================
print(f"\n{'='*50}")
print(f"  DISTRIBUTION SENTIMENT")
print(f"{'='*50}")
dist = df_assigned['sentiment'].value_counts()
for label, count in dist.items():
    print(f"  {label:<12} : {count:>7,}  ({count/len(df_assigned)*100:.1f}%)")

print(f"\n  Score moyen par aspect :")
print(f"  {'Aspect':<30} {'%Pos':>6}  {'%Nég':>6}  {'Score ABSA':>10}")
print(f"  {'─'*55}")
for key, label in ASPECT_LABELS.items():
    sub = df_assigned[df_assigned['aspect'] == key]
    if len(sub) == 0:
        continue
    pos   = (sub['sentiment'] == 'positive').sum()
    neg   = (sub['sentiment'] == 'negative').sum()
    total = len(sub)
    score = round((pos - neg) / total * 100, 1)
    print(f"  {label:<30} {pos/total*100:>5.1f}%  {neg/total*100:>5.1f}%  {score:>+10.1f}")

# ============================================================
# SAUVEGARDE
# ============================================================
cols = [c for c in [
    'avis_id', 'phrase', 'topic', 'topic_keywords',
    'aspect', 'aspect_label', 'source_aspect', 'cosine_score',
    'hotel_name', 'hotel_city', 'source', 'langue', 'note',
    'date_sejour', 'saison', 'type_voyage',
    'sentiment', 'score_sentiment',
] if c in df_assigned.columns]

df_assigned[cols].to_csv('/kaggle/working/df_final_absa_v4.csv', index=False, encoding='utf-8-sig')
print(f"\n✓ Sauvegardé → /kaggle/working/df_final_absa_v4.csv")
print(f"  {len(df_assigned):,} phrases  |  {df_assigned['hotel_name'].nunique()} hôtels  |  {df_assigned['aspect'].nunique()} aspects")

Phrases à analyser : 102,465


config.json:   0%|          | 0.00/928 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/451 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Analyse sentiment (102,465 phrases, batch=512)...
   10,000 / 102,465  (9.8%)
   20,000 / 102,465  (19.5%)
   30,000 / 102,465  (29.3%)
   40,000 / 102,465  (39.0%)
   50,000 / 102,465  (48.8%)
   60,000 / 102,465  (58.6%)
   70,000 / 102,465  (68.3%)
   80,000 / 102,465  (78.1%)
   90,000 / 102,465  (87.8%)
  100,000 / 102,465  (97.6%)

  DISTRIBUTION SENTIMENT
  positive     :  52,252  (51.0%)
  neutral      :  28,318  (27.6%)
  negative     :  21,895  (21.4%)

  Score moyen par aspect :
  Aspect                           %Pos    %Nég  Score ABSA
  ───────────────────────────────────────────────────────
  Personnel                       70.5%   14.6%       +55.9
  Service                         60.8%   17.0%       +43.8
  Chambre & Propreté              44.9%   27.5%       +17.4
  Restauration                    47.2%   22.3%       +24.9
  Localisation                    42.7%    5.7%       +37.0
  Expérience & Valeur             75.9%   11.7%       +64.2
  Rapport qualité-prix     

In [26]:
!pip install xlsxwriter -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.3 MB/s eta 0:00:0000:01


## Validation manuelle sur 200 avis annotés

Évaluation de la fiabilité du pipeline : 200 avis tirés du corpus, annotés et 
vérifiés manuellement (gold standard), servant uniquement de jeu de test. 
Comparaison des prédictions du modèle aux annotations (accuracy, précision, 
rappel, F1, Cohen's Kappa) pour l'affectation des aspects et la classification du sentiment.

In [27]:
import pandas as pd

# ── Distribution proportionnelle 200 phrases ─────────────────
SAMPLE_DISTRIBUTION = {
    'Français'    : 80,
    'Anglais'     : 60,
    'Espagnol'    : 15,
    'Italien'     : 10,
    'Allemand'    : 10,
    'Néerlandais' : 8,
    'Portugais'   : 8,
    'Arabe'       : 5,
    'Russe'       : 4,
}
# Total = 200

# ── Échantillonnage stratifié ────────────────────────────────
samples = []
for langue, n in SAMPLE_DISTRIBUTION.items():
    subset = df_assigned[df_assigned['langue'] == langue]
    n_real = min(n, len(subset))  # sécurité si pas assez de phrases
    samples.append(subset.sample(n=n_real, random_state=42))

df_sample = pd.concat(samples).sample(frac=1, random_state=42).reset_index(drop=True)
df_sample.index = df_sample.index + 1  # commencer à 1

# ── Colonnes pour validation manuelle ───────────────────────
df_validation = df_sample[[
    'phrase', 'langue', 'aspect', 'aspect_label',
    'sentiment', 'score_sentiment', 'hotel_name', 'hotel_city'
]].copy()

df_validation['aspect_correct']    = ''
df_validation['aspect_reel']       = ''
df_validation['sentiment_correct'] = ''
df_validation['sentiment_reel']    = ''

# ── Stats ────────────────────────────────────────────────────
print(f"Total : {len(df_validation)} phrases\n")
print("Distribution par langue :")
print(df_validation['langue'].value_counts())
print("\nDistribution par aspect :")
print(df_validation['aspect'].value_counts())

# ── Export Excel avec mise en forme ─────────────────────────
output_path = '/kaggle/working/validation_manuelle_200.xlsx'

with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    df_validation.to_excel(writer, sheet_name='Validation', index=True)

    wb  = writer.book
    ws  = writer.sheets['Validation']

    # Formats
    header_fmt = wb.add_format({'bold': True, 'bg_color': '#2F75B6',
                                 'font_color': 'white', 'border': 1,
                                 'text_wrap': True, 'align': 'center'})
    oui_fmt    = wb.add_format({'bg_color': '#C6EFCE', 'border': 1})
    non_fmt    = wb.add_format({'bg_color': '#FFC7CE', 'border': 1})
    phrase_fmt = wb.add_format({'text_wrap': True, 'border': 1, 'valign': 'top'})
    cell_fmt   = wb.add_format({'border': 1, 'valign': 'top'})
    pos_fmt    = wb.add_format({'bg_color': '#E2EFDA', 'border': 1})
    neg_fmt    = wb.add_format({'bg_color': '#FCE4D6', 'border': 1})
    neu_fmt    = wb.add_format({'bg_color': '#FFF2CC', 'border': 1})

    # Largeurs colonnes
    ws.set_column('A:A', 6)    # index
    ws.set_column('B:B', 55)   # phrase
    ws.set_column('C:C', 12)   # langue
    ws.set_column('D:D', 20)   # aspect
    ws.set_column('E:E', 22)   # aspect_label
    ws.set_column('F:F', 12)   # sentiment
    ws.set_column('G:G', 8)    # score
    ws.set_column('H:H', 20)   # hotel_name
    ws.set_column('I:I', 15)   # hotel_city
    ws.set_column('J:J', 14)   # aspect_correct
    ws.set_column('K:K', 20)   # aspect_reel
    ws.set_column('L:L', 16)   # sentiment_correct
    ws.set_column('M:M', 16)   # sentiment_reel

    # Hauteur lignes
    ws.set_default_row(40)
    ws.set_row(0, 30)

    # Colorer sentiment prédit
    for row_num in range(1, len(df_validation) + 1):
        sent = df_validation.iloc[row_num - 1]['sentiment']
        fmt  = pos_fmt if sent == 'positive' else (neg_fmt if sent == 'negative' else neu_fmt)
        ws.write(row_num, 5, sent, fmt)

    # Feuille guide
    guide = wb.add_worksheet('Guide')
    guide.write(0, 0, 'GUIDE DE VALIDATION', wb.add_format({'bold': True, 'font_size': 14}))
    guide.write(2, 0, 'aspect_correct / sentiment_correct :')
    guide.write(3, 0, '  → Oui  : le modèle a bien prédit')
    guide.write(4, 0, '  → Non  : le modèle a mal prédit')
    guide.write(6, 0, 'aspect_reel (seulement si aspect_correct = Non) :')
    for i, (k, v) in enumerate(ASPECT_LABELS.items()):
        guide.write(7 + i, 0, f'  {k}  →  {v}')
    guide.write(19, 0, 'sentiment_reel (seulement si sentiment_correct = Non) :')
    guide.write(20, 0, '  positive  /  negative  /  neutral')
    guide.set_column('A:A', 50)

print(f"\n✅ Fichier sauvegardé → {output_path}")
print(f"   Télécharge depuis Kaggle : Output → validation_manuelle_200.xlsx")

Total : 200 phrases

Distribution par langue :
langue
Français       80
Anglais        60
Espagnol       15
Allemand       10
Italien        10
Néerlandais     8
Portugais       8
Arabe           5
Russe           4
Name: count, dtype: int64

Distribution par aspect :
aspect
chambre_proprete    61
restauration        32
personnel           32
localisation        24
service             14
equipements         11
experience           9
bruit                7
temperature          6
wifi                 3
prix                 1
Name: count, dtype: int64

✅ Fichier sauvegardé → /kaggle/working/validation_manuelle_200.xlsx
   Télécharge depuis Kaggle : Output → validation_manuelle_200.xlsx


In [29]:
import pandas as pd
from sklearn.metrics import f1_score, recall_score, cohen_kappa_score

df = pd.read_excel('/kaggle/input/datasets/omarboukellalfsfl/validation-manuelle-200/validation_manuelle_200_annotee.xlsx')

df['aspect_final']    = df.apply(lambda r: r['aspect_reel'] if r['aspect_correct'] == 'Non' else r['aspect'], axis=1)
df['sentiment_final'] = df.apply(lambda r: r['sentiment_reel'] if r['sentiment_correct'] == 'Non' else r['sentiment'], axis=1)

for nom, yt, yp in [
    ('ASPECT',    df['aspect_final'],    df['aspect']),
    ('SENTIMENT', df['sentiment_final'], df['sentiment']),
]:
    f1  = f1_score(yt, yp, average='weighted', zero_division=0)
    rec = recall_score(yt, yp, average='weighted', zero_division=0)
    kap = cohen_kappa_score(yt, yp)
    print(f"{nom:<12} → F1: {f1:.3f}  |  Recall: {rec:.3f}  |  Kappa: {kap:.3f}")

ASPECT       → F1: 0.784  |  Recall: 0.780  |  Kappa: 0.748
SENTIMENT    → F1: 0.878  |  Recall: 0.880  |  Kappa: 0.817
